In [1]:
import paddle
print(paddle.utils.run_check())

/media/data3/users/huytq/miniconda3/envs/huy/lib/python3.11/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Running verify PaddlePaddle program ... 


/media/data3/users/huytq/miniconda3/envs/huy/lib/python3.11/site-packages/paddle/pir/math_op_patch.py:241: UserWarning: Tensor do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(
I0316 02:35:46.326020 1807799 pir_interpreter.cc:1529] New Executor is Running ...
I0316 02:35:46.326313 1807799 pir_interpreter.cc:1552] pir interpreter is running by multi-thread mode ...


PaddlePaddle works well on 1 CPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.
None


In [ ]:
from pathlib import Path
import subprocess
import sys

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
SCRIPT_PATH = BASE_DIR / "ppstructurev3_to_md.py"
OUT_DIR = BASE_DIR / "outputs_ppstructurev3_md"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f"Khong tim thay script: {SCRIPT_PATH}")
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Khong tim thay thu muc data: {DATA_DIR}")

pdf_files = sorted(DATA_DIR.rglob("*.pdf"))
print(f"Tim thay {len(pdf_files)} file PDF trong {DATA_DIR}")

for i, pdf_path in enumerate(pdf_files, 1):
    cmd = [
        sys.executable,
        str(SCRIPT_PATH),
        "--input", str(pdf_path),
        "--output_dir", str(OUT_DIR),
        "--lang", "vi",
        "--device", "cpu",
        "--disable_mkldnn",
        "--page_header",
    ]

    print(f"[{i}/{len(pdf_files)}] Dang xu ly: {pdf_path.name}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print("  OK")
    else:
        print("  FAIL")
        print(result.stderr[-1200:])

print(f"\nXong. File markdown o: {OUT_DIR}")

Tim thay 11 file PDF
[1/11] Dang xu ly: Public001.pdf
  OK  -> Public001_paddle_det_vietocr.txt
[2/11] Dang xu ly: Public002.pdf
  OK  -> Public002_paddle_det_vietocr.txt
[3/11] Dang xu ly: Public003.pdf
  OK  -> Public003_paddle_det_vietocr.txt
[4/11] Dang xu ly: Public004.pdf
  OK  -> Public004_paddle_det_vietocr.txt
[5/11] Dang xu ly: Public005.pdf
  OK  -> Public005_paddle_det_vietocr.txt
[6/11] Dang xu ly: Public253.pdf
  OK  -> Public253_paddle_det_vietocr.txt
[7/11] Dang xu ly: Public257.pdf


In [2]:
from pathlib import Path
import subprocess
import sys

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data_2"
SCRIPT_PATH = BASE_DIR / "ppstructurev3_to_md.py"
OUTPUT_DIR = BASE_DIR / "outputs_ppstructurev3_md"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f"Khong tim thay script: {SCRIPT_PATH}")
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Khong tim thay thu muc data: {DATA_DIR}")

pdf_files = sorted(DATA_DIR.rglob("*.pdf"))
print(f"Tim thay {len(pdf_files)} file PDF trong {DATA_DIR}")

success_files = []
failed_files = []

for index, pdf_path in enumerate(pdf_files, 1):
    cmd = [
        sys.executable,
        str(SCRIPT_PATH),
        "--input", str(pdf_path),
        "--output_dir", str(OUTPUT_DIR),
        "--lang", "vi",
        "--device", "cpu",
        "--vietocr_device", "cpu",
        "--disable_mkldnn",
        "--page_header",
    ]

    print(f"[{index}/{len(pdf_files)}] Dang xu ly: {pdf_path}")
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        success_files.append(pdf_path)
        print("  OK")
    else:
        failed_files.append(pdf_path)
        print("  FAIL")
        if result.stderr:
            print(result.stderr[-1500:])
        elif result.stdout:
            print(result.stdout[-1500:])

print("\nTong ket")
print(f"- Thanh cong: {len(success_files)}")
print(f"- That bai: {len(failed_files)}")
print(f"- Thu muc output: {OUTPUT_DIR}")

if failed_files:
    print("\nDanh sach file loi:")
    for failed_path in failed_files:
        print(f"- {failed_path}")

Tim thay 1 file PDF trong /media/data3/users/huytq/huy/data_2
[1/1] Dang xu ly: /media/data3/users/huytq/huy/data_2/Public253.pdf


KeyboardInterrupt: 

In [3]:
from pathlib import Path
import shutil
import subprocess
import sys
import time

import pypdfium2 as pdfium

BASE_DIR = Path.cwd()
PDF_PATH = BASE_DIR / "data_2" / "Public253.pdf"
SCRIPT_PATH = BASE_DIR / "ppstructurev3_to_md.py"
PAGE_IMG_DIR = BASE_DIR / "debug_public253_pages"
PAGE_OUT_DIR = BASE_DIR / "outputs_ppstructurev3_md" / "Public253_pages"

# Dat None de chay tat ca cac trang, hoac vi du [1, 2, 5]
PAGE_NUMBERS = None
SCALE = 2.0

if not PDF_PATH.exists():
    raise FileNotFoundError(f"Khong tim thay file PDF: {PDF_PATH}")
if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f"Khong tim thay script: {SCRIPT_PATH}")

if PAGE_IMG_DIR.exists():
    shutil.rmtree(PAGE_IMG_DIR)
PAGE_IMG_DIR.mkdir(parents=True, exist_ok=True)
PAGE_OUT_DIR.mkdir(parents=True, exist_ok=True)

pdf = pdfium.PdfDocument(str(PDF_PATH))
total_pages = len(pdf)
selected_pages = PAGE_NUMBERS or list(range(1, total_pages + 1))

print(f"Tong so trang: {total_pages}")
print(f"Se chay cac trang: {selected_pages}")

page_timings = []
failed_pages = []

for page_number in selected_pages:
    page_index = page_number - 1
    page = pdf[page_index]
    bitmap = page.render(scale=SCALE)
    pil_image = bitmap.to_pil()

    page_img_path = PAGE_IMG_DIR / f"Public253_page_{page_number:03d}.png"
    pil_image.save(page_img_path)

    start_time = time.time()
    cmd = [
        sys.executable,
        str(SCRIPT_PATH),
        "--input", str(page_img_path),
        "--output_dir", str(PAGE_OUT_DIR),
        "--lang", "vi",
        "--device", "cpu",
        "--vietocr_device", "cpu",
        "--disable_mkldnn",
        "--page_header",
    ]

    print(f"\nDang xu ly trang {page_number}: {page_img_path.name}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - start_time

    if result.returncode == 0:
        print(f"  OK - {elapsed:.1f}s")
        page_timings.append((page_number, elapsed))
    else:
        print(f"  FAIL - {elapsed:.1f}s")
        failed_pages.append(page_number)
        if result.stderr:
            print(result.stderr[-1500:])
        elif result.stdout:
            print(result.stdout[-1500:])

print("\nTong ket theo trang")
for page_number, elapsed in page_timings:
    print(f"- Trang {page_number}: {elapsed:.1f}s")

if failed_pages:
    print("\nTrang loi:")
    for page_number in failed_pages:
        print(f"- Trang {page_number}")

print(f"\nAnh tung trang nam o: {PAGE_IMG_DIR}")
print(f"Markdown tung trang nam o: {PAGE_OUT_DIR}")

Tong so trang: 22
Se chay cac trang: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]

Dang xu ly trang 1: Public253_page_001.png
  OK - 243.9s

Dang xu ly trang 2: Public253_page_002.png
  OK - 291.8s

Dang xu ly trang 3: Public253_page_003.png
  OK - 76.5s

Dang xu ly trang 4: Public253_page_004.png
  OK - 233.5s

Dang xu ly trang 5: Public253_page_005.png
  OK - 193.5s

Dang xu ly trang 6: Public253_page_006.png
  OK - 322.8s

Dang xu ly trang 7: Public253_page_007.png
  OK - 298.5s

Dang xu ly trang 8: Public253_page_008.png
  OK - 281.6s

Dang xu ly trang 9: Public253_page_009.png
  OK - 295.4s

Dang xu ly trang 10: Public253_page_010.png
  OK - 282.8s

Dang xu ly trang 11: Public253_page_011.png
  OK - 317.3s

Dang xu ly trang 12: Public253_page_012.png
  OK - 300.6s

Dang xu ly trang 13: Public253_page_013.png
  OK - 296.3s

Dang xu ly trang 14: Public253_page_014.png
  OK - 294.8s

Dang xu ly trang 15: Public253_page_015.png
  OK - 298.2s

Dang xu